# Length and offset sweeps

TDECQ, semi-analytic SER and channel bandwidth versus fiber length and connector offset for the default link.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightstone

In [ ]:
sim = lightstone.Simulator()
L = np.arange(0, 251, 25.0); td, ser, bw = [], [], []
for l in L:
    r = sim.set(fiber__length=float(l)).run()
    t = r.tdecq()["tdecq_db"]; td.append(np.nan if t is None else t)
    ser.append(r.analyze()["ser"]); b = r.metrics["channel_bandwidth"]; bw.append(np.nan if b is None else b / 1e9)  # None: no roll-off (L = 0)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(L, td, "o-"); ax[0].axhline(4.5, color="r", ls="--", label="example TDECQ limit"); ax[0].set(xlabel="length [m]", ylabel="TDECQ [dB]"); ax[0].legend()
ax[1].semilogy(L, ser, "o-"); ax[1].set(xlabel="length [m]", ylabel="SER (5-tap FFE)")
ax[2].plot(L, bw, "o-"); ax[2].set(xlabel="length [m]", ylabel="channel BW [GHz]"); plt.tight_layout()

The 4.5 dB line is only a visual reference; check the applicable PMD clause for the actual limit.

In [ ]:
sim = lightstone.Simulator({"fiber.length": 100.0})
off = np.arange(0, 21, 2.0); td, emb = [], []
for o in off:
    r = sim.set(launch__offset=o * 1e-6).run()
    t = r.tdecq()["tdecq_db"]; td.append(np.nan if t is None else t); emb.append(r.metrics["emb_length"] / 1e9 if r.metrics["emb_length"] else np.nan)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(off, td, "o-"); ax[0].set(xlabel="offset [µm]", ylabel="TDECQ @100 m [dB]")
ax[1].plot(off, emb, "o-"); ax[1].set(xlabel="offset [µm]", ylabel="modal BW [MHz·km]"); plt.tight_layout()

In [ ]:
r = lightstone.simulate({"fiber.length": 100.0})
a = r.analyze(); counts, meta = r.eye(analysis=a)
plt.figure(figsize=(7, 4)); plt.imshow(np.log1p(counts), aspect="auto", cmap="inferno", extent=[-1, 1, meta["v_min"]*1e3, meta["v_max"]*1e3])
plt.xlabel("time [UI]"); plt.ylabel("TIA output [mV]"); plt.title("Eye @ 100 m (after the 5-tap receiver FFE)");